In [1]:
# with G > 0, B > 0, Exp-shape, Cs
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Prevent tensorflow I: and W:
import gc; gc.collect()                             # clear GPU memory

import math as math
import tensorflow as tf
from tensorflow.python.ops import math_ops
import numpy as np
import scipy
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib import pyplot
%matplotlib auto
import time
from scipy.interpolate import CubicSpline

#plt.ion()                                           # dynamic graph
#fig1 = plt.figure(figsize=(10,8))
#fig2 = plt.figure(figsize=(8,5))
#fig3 = plt.figure(figsize=(8,5))
#fig4 = plt.figure(figsize=(8,5))
#fig5 = plt.figure(figsize=(8,5))

Using matplotlib backend: <object object at 0x7fed87739cc0>


In [2]:
# Проверка интеграла
x = np.linspace(0, 100, num=1_000_000)
y = np.ones(x.shape)

In [3]:
# Alghoritm's constants
n_grid, n_grid2, iterations, rewrite_file = 40000, 100, 80000, True # n_grid - main grid; n_grid2 - small grid for slow integral calcs

# Process constants
Th0, Lmin, Lmax, tmax = 28.2417878478347, 0.001, 350.0, 48.0 # Th = ln(u), u = dN/dL - numeric density in the interval
Thnuc_f = 12.0                                               # Nucleation Rate Thnuc = ln(B_nuc)
rho = tf.constant(2420.0, dtype=tf.float32)                  # hydrate density
Al2O3_in, Na2Oo, Na2Ok, TC = 120.0, 157.5, 140.0, 62.0       # Liquor at the Head Tank
MW_Al2O3, MW_AlOH3 = 0.102, 0.078                            # Molar weights of Al2O3 & Al[OH]3, kg/mol

G = tf.constant(1.0/24, dtype=tf.float32)
BminusD = tf.constant(0.00, dtype=tf.float32)

# Normalization limits
Th_min, Th_max =    10.0,    30.0
m2_min, m2_max =  5000.0, 15000.0
m3_min, m3_max =     0.2,     0.8
A_min,  A_max  =    30.0,   160.0
dA_min, dA_max  = -120.0,     0.0
norm_func = lambda x,a,b: (x-a)/(b-a)

# initial distribution of particle density: ln(u0)
Th0_exp_f = lambda x: Th0 * np.exp(-x/393.265139450887)
[Th0_exp_f(Lmin), Th0_exp_f(Lmax)]

[28.24171603431983, 11.597828829610375]

In [4]:
# Main grid L[i+1] = L[i]*2^(1/3/q)
q = n_grid/3*np.log(2)/np.log(Lmax/Lmin)
def qgrid_func(n, random):
    if (random==False):
        x = np.reshape(np.linspace(0, n, n), [n, 1])
    else:
        x = tf.random.uniform((n, 1), dtype=tf.float32, minval=0, maxval=n)
    return tf.Variable(np.exp(np.log(Lmin) + x/3/q*np.log(2)), tf.float32)
#x = qgrid_func(n=n_grid, random=True)
q

723.9689779078333

In [5]:
@tf.function
def Aeq_func(Na2Oo, Na2Ok, TC):
    # Returns Equilibrium concentration of Al2O3 in solution, g|L
    # where: Na2Oo - Total Titrated Soda as Na2O, g|L; Na2Ok - Caustic Soda as Na2O, g|L; TC - temperature, C deg.
    S = 1 + 0.026*(TC-50) + 0.000127*(TC-50)**2
    T = 1 + 0.013*(Na2Ok-140) + 0.0000642*(Na2Ok-140)**2
    U = 0.134*(Na2Oo-Na2Ok)
    V = 5.82 + 0.07566*(Na2Ok-140)
    return 51.7*S*T + U + V
Aeq_func(Na2Oo=157.5, Na2Ok=140, TC=58).numpy()

71.03882

In [6]:
@tf.function
def G_func(Al2O3, Na2Oo, Na2Ok, TC):
    # Returns Growth rate of alumium hydrohide Al[OH]3, um/hour
    # where: Al2O3 - concentration of Al2O3, g|L; Na2Oo - Total Titrated Soda as Na2O, g|L; Na2Ok - Caustic Soda as Na2O, g|L; TC - temperature, C deg.
    k0, Ea, n = 3.7e12/24, 64000.0, 2.0
    Aeq = Aeq_func(Na2Oo, Na2Ok, TC)
    dA = ((Al2O3-Aeq)+tf.abs(Al2O3-Aeq))/2 # only positive values are available
    G = k0 * tf.exp(-Ea/8.314/(TC+273.15)) * (dA/(1.71*Na2Ok))**n
    return G
G_func(Al2O3=85.0, Na2Oo=157.5, Na2Ok=140, TC=62).numpy()

0.018504087

In [7]:
# function for trapezoidal integral calculation
@tf.function
def trapezoidal_integral_approx(x, y):
    return -1*math_ops.reduce_sum(
            math_ops.multiply(x[:-1] - x[1:],
                              (y[:-1] + y[1:]) / 2.))
    
@tf.function                   # Function returns 2-nd and 3-rd moments of population at any point 
def m2m3_func(t, x):           # by intergation of Th at the net1-output, where t[h], x[um] - tensors.
    nx = 1000                  # User defined enough number of points for integration
    nt = tf.size(t)            # Length of requested output
    m2_t = tf.TensorArray(tf.float32, size=0, dynamic_size=True)
    m3_t = tf.TensorArray(tf.float32, size=0, dynamic_size=True)
    ls = tf.linspace(0.0, 1.0, nx)
    i=0
    while i < nt:
        # make t-tensor with t[i] & nx points x-tensor
        ti_ = tf.slice(t, begin=[i,0], size=[1,1])
        ti  = tf.ones(nx, 1)*ti_
        ti  = tf.reshape(ti, [nx, 1])
        xi_ = tf.slice(x, begin=[i,0], size=[1,1])
        xi  = Lmin+(ls*(xi_-Lmin))
        xi  = tf.reshape(xi, [nx, 1])
        Th = net1(tf.concat([ti, xi], axis=1))
        u, Li = tf.exp(Th), xi*1e-6
        dm2 = tf.math.multiply(u, Li**2)
        dm3 = tf.math.multiply(u, Li**3)
        m2_t = m2_t.write(i,trapezoidal_integral_approx(xi,dm2))
        m3_t = m3_t.write(i,trapezoidal_integral_approx(xi,dm3))
        i+=1
    m2_t = m2_t.stack()
    m3_t = m3_t.stack()
    return [m2_t, m3_t]    

# initial moments (t=0)  
m2dL_in_func= lambda x: tf.exp(Th0_exp_f(x))*(x*1e-6)**2              # function under integral
m3dL_in_func= lambda x: tf.exp(Th0_exp_f(x))*(x*1e-6)**3              # function under integral
x = tf.Variable(tf.linspace(Lmin, Lmax, 1000))                     # some x distribution
m2_in = trapezoidal_integral_approx(tf.Variable(tf.linspace(Lmin, Lmax, 1000)), m2dL_in_func(x)).numpy()
m3_in = trapezoidal_integral_approx(tf.Variable(tf.linspace(Lmin, Lmax, 1000)), m3dL_in_func(x)).numpy()
print ("SSL init = ", math.pi*m2_in)
print ("Cs init = ", (math.pi/6*rho*m3_in).numpy())

SSL init =  39681.285001644435
Cs init =  794.78534


In [8]:
# model definition
class PDE_Model(tf.keras.Model):
    def __init__(self, NN):
        super(PDE_Model, self).__init__()
        self.input_layer = tf.keras.layers.Dense(NN, input_dim= 2, activity_regularizer=tf.keras.regularizers.L2(0.0001))
        self.dropout = tf.keras.layers.Dropout(0.3)  
        self.hidden_layer = tf.keras.layers.Dense(NN, activity_regularizer=tf.keras.regularizers.L2(0.0001))
        #self.hidden_layer.trainable = False
        self.dropout = tf.keras.layers.Dropout(0.3)    
        self.hidden_layer_2 = tf.keras.layers.Dense(NN, activity_regularizer=tf.keras.regularizers.L2(0.0001))
        #self.hidden_layer_2.trainable = False
        self.dropout = tf.keras.layers.Dropout(0.3)           
        self.output_layer = tf.keras.layers.Dense(1, activity_regularizer=tf.keras.regularizers.L2(0.0001))      
    def call(self, x):
        out = tf.tanh(self.input_layer(x))
        out = tf.tanh(self.hidden_layer(out))
        out = tf.tanh(self.hidden_layer_2(out))
        out = self.output_layer(out)
        return out   
        
class ODE_Model(tf.keras.Model):
    def __init__(self, NN):
        super(ODE_Model, self).__init__()
        self.input_layer = tf.keras.layers.Dense(NN, input_dim= 1, activity_regularizer=tf.keras.regularizers.L2(0.0001))    
        self.dropout = tf.keras.layers.Dropout(0.3)          
        self.hidden_layer = tf.keras.layers.Dense(NN, activity_regularizer=tf.keras.regularizers.L2(0.0001))
        #self.hidden_layer.trainable = False
        self.dropout = tf.keras.layers.Dropout(0.3)        
        self.hidden_layer_2 = tf.keras.layers.Dense(NN, activity_regularizer=tf.keras.regularizers.L2(0.0001))
        #self.hidden_layer_2.trainable = False
        self.dropout = tf.keras.layers.Dropout(0.3)        
        self.output_layer = tf.keras.layers.Dense(1, activity_regularizer=tf.keras.regularizers.L2(0.0001))        
    def call(self, x):
        out = tf.tanh(self.input_layer(x))
        out = tf.tanh(self.hidden_layer(out))
        out = tf.tanh(self.hidden_layer_2(out))
        out = self.output_layer(out)
        return out  

net1 = PDE_Model(256) # [Th] = f(t, L)
net2 = ODE_Model(64) # [m2] = f(t)
net3 = ODE_Model(64)  # [m3] = f(t)
net4 = ODE_Model(64)  # [A]  = f(t)

# loss function
mse = tf.keras.losses.MeanSquaredError()
# optimizer
optimizer1 = tf.keras.optimizers.Adam(learning_rate=1e-4)
optimizer2 = tf.keras.optimizers.Adam(learning_rate=1e-3)
optimizer3 = tf.keras.optimizers.Adam(learning_rate=1e-3)
optimizer4 = tf.keras.optimizers.Adam(learning_rate=1e-3)
# metrics
train_acc_metric1 = tf.keras.metrics.MeanSquaredError()
train_acc_metric2 = tf.keras.metrics.MeanSquaredError()
train_acc_metric3 = tf.keras.metrics.MeanSquaredError()
train_acc_metric4 = tf.keras.metrics.MeanSquaredError()
# pack
net1.compile(optimizer1, loss=mse)
net2.compile(optimizer2, loss=mse)
net3.compile(optimizer3, loss=mse)
net4.compile(optimizer4, loss=mse)
# build models and load pre-trained weights
net1(tf. concat([tf.Variable(tf.random.uniform((10, 1), dtype=tf.float32, minval=0.0, maxval=tmax)),
                 tf.Variable(tf.random.uniform((10, 1), dtype=tf.float32, minval=Lmin, maxval=Lmax))], axis=1))
net2(tf. concat([tf.Variable(tf.random.uniform((10, 1), dtype=tf.float32, minval=0.0, maxval=tmax))], axis=1))
net3(tf. concat([tf.Variable(tf.random.uniform((10, 1), dtype=tf.float32, minval=0.0, maxval=tmax))], axis=1))
net4(tf. concat([tf.Variable(tf.random.uniform((10, 1), dtype=tf.float32, minval=0.0, maxval=tmax))], axis=1))
net1.load_weights('net_pbe//net_pbe_20_Th_v9.h5')
net2.load_weights('net_pbe//net_pbe_20_m2_v9.h5')
net3.load_weights('net_pbe//net_pbe_20_m3_v9.h5')
net4.load_weights('net_pbe//net_pbe_20_A_v9.h5')

In [9]:
start_time = time.time()
epochs, losses1, losses2, losses3, losses4 = [], [], [], [], []

# Initialize constants
t0 = tf.constant(np.zeros((n_grid, 1)), dtype=tf.float32)                  # t0 = 0
x_min_bc = tf.constant(np.ones((n_grid, 1))*Lmin, dtype=tf.float32)        # x_min = Lmin
x_max_bc = tf.constant(np.ones((n_grid, 1))*Lmax, dtype=tf.float32)        # x_max = Lmax
x_max2_bc = tf.Variable(tf.random.uniform((n_grid2, 1), dtype=tf.float32, minval=Lmax, maxval=Lmax))

zeros_vector = tf.constant(np.zeros((n_grid, 1)), dtype=tf.float32)        # zeros
zeros_vector2 = tf.constant(np.zeros((n_grid2, 1)), dtype=tf.float32)

for epoch in range(iterations):

    # Initialize constants
    dTh_dt_nuc = tf.constant(np.ones((n_grid, 1))*Thnuc_f, dtype=tf.float32)   # boundary dTh/dt=Thnuc    
    
    # Initialize variables
    t_var = tf.Variable(tf.random.uniform((n_grid, 1), dtype=tf.float32, minval=0.0, maxval=tmax))
    
    x_var = qgrid_func(n=n_grid, random=True)
    L = x_var*1e-6
    
    t_var2 = tf.Variable(tf.random.uniform((n_grid2, 1), dtype=tf.float32, minval=0.0, maxval=tmax))
    moments = m2m3_func(t_var2, x_max2_bc) # it is the slowest line of the whole cycle because of integral for moments
    m2_field2, m3_field2 = tf.reshape(moments[0], [n_grid2, 1]), tf.reshape(moments[1], [n_grid2, 1])
    
    # Gradients
    with tf. GradientTape(persistent=True) as tape:                                    # gradients of loss function
        Th_ic = net1(tf.concat([t0, x_var], axis=1))                                   # initial conditions of Th
        m2_ic = net2(tf.concat([t0], axis=1))                                          # initial conditions of m2
        m3_ic = net3(tf.concat([t0], axis=1))                                          # initial conditions of m3
        A_ic  = net4(tf.concat([t0], axis=1))                                          # initial conditions of A
        with tf.GradientTape(persistent=True) as tape_xx:                              # gradients for second-order derivations
            with tf.GradientTape(persistent=True) as tape_x:                           # gradients for first-order derivations
                Th_hat = net1(tf.concat([t_var, x_var], axis=1))                       # Th
                Th_min_bc = net1(tf.concat([t_var, x_min_bc], axis=1))                 # Th at Lmin_bc
                Th_max_bc = net1(tf.concat([t_var, x_max_bc], axis=1))                 # Th at Lmax_bc  
                m2_hat = net2(tf.concat([t_var2], axis=1))                             # m2 (at the short field)          
                m3_hat = net3(tf.concat([t_var2], axis=1))                             # m3 (at the short field)     
                A_hat = net4(tf.concat([t_var], axis=1))                               # A
            dTh_dt = tape_x.gradient(Th_hat, t_var)                                    # dTh/dt
            dTh_dx = tape_x.gradient(Th_hat, x_var)                                    # dTh/dL
            dTh_dt_min_bc = tape_x.gradient(Th_min_bc, t_var)                          # dTh/dt at Lmin_bc
            dTh_dt_max_bc = tape_x.gradient(Th_max_bc, t_var)                          # dTh/dt at Lmax_bc
            #dCm2_dt = tape_x.gradient(m2_hat, t_var2)                                 # dCs/dt
            dA_dt = tape_x.gradient(A_hat, t_var)                                      # dA/dt
        #du_dxx = tape_xx. gradient(du_dx, x_var)   
        
        # --------- First - PDE for Th ------------------        
        #eq(1) - ic Th
        eq1_1 = Th_ic                                        # achieved
        eq1_2 = Th0_exp_f(x_var)                             # required
        mse_1 = mse(eq1_1, eq1_2)         
        
        #eq(2) - pde Th
        #eq2_1 = dTh_dt + G_func(A_hat, Na2Oo, Na2Ok, TC) * dTh_dx #- (BminusD)/tf.exp(Th_hat)  # achieved
        eq2_1 = dTh_dt + G * dTh_dx #- (BminusD)/tf.exp(Th_hat)  # achieved
        mse_2 = mse(eq2_1, zeros_vector)

        #eq(3) - bc(xmin) Th
        eq3_1 = dTh_dt_min_bc - tf.exp(dTh_dt_nuc - Th_hat)  # achieved
        eq3_2 = zeros_vector                                 # required (nucleation)
        mse_3 = mse(eq3_1, eq3_2)

        #eq(4) - bc(xmax) Th
        eq4_1 = dTh_dt_max_bc                                # achieved
        mse_4 = mse(eq4_1, zeros_vector)
        
        # --------- Second - ODE for m2 ------------------                                       
        #eq(5) - ic Cs
        eq5_1 = m2_ic                                        # achieved
        eq5_2 = tf.constant(m2_in, dtype=tf.float32)         # required
        mse_5 = mse(norm_func(eq5_1, m2_min, m2_max), norm_func(eq5_2, m2_min, m2_max))
        
        #eq(6) - ode Cs 
        eq6_1 = m2_hat                                       # achieved
        eq6_2 = m2_field2                                    # required
        mse_6 = mse(norm_func(eq6_1, m2_min, m2_max), norm_func(eq6_2, m2_min, m2_max))
   
        # --------- Third - ODE for m3 ------------------                                       
        #eq(5) - ic Cs
        eq7_1 = m3_ic                                        # achieved
        eq7_2 = tf.constant(m3_in, dtype=tf.float32)         # required
        mse_7 = mse(norm_func(eq7_1, m3_min, m3_max), norm_func(eq7_2, m3_min, m3_max))/n_grid2

        #eq(6) - ode Cs 
        eq8_1 = m3_hat                                       # achieved
        eq8_2 = m3_field2                                    # required
        mse_8 = mse(norm_func(eq8_1, m3_min, m3_max), norm_func(eq8_2, m3_min, m3_max))
        #= -1/(MW_Al2O3/(2*MW_AlOH3) * (rho/(rho-Cs_hat))**2 ) * dA_dt  # required       

        # --------- Fourth - ODE for A ------------------
        #eq(9) - ic A
        eq9_1 = A_ic                                         # achieved
        eq9_2 = tf.constant(Al2O3_in, dtype=tf.float32)      # required
        mse_9 = mse(norm_func(eq9_1, A_min, A_max), norm_func(eq9_2, A_min, A_max))
        
        #eq(10) - ode A
        eq10_1 = dA_dt                                                                      # achieved
        eq10_2 = -MW_Al2O3/(2*MW_AlOH3) * G_func(A_hat, Na2Oo, Na2Ok, TC) * 40.0            # required, where SSL = 40.0   
        mse_10 = mse(norm_func(eq10_1, dA_min, dA_max), norm_func(eq10_2, dA_min, dA_max))

        
        loss1 = mse_1 + mse_2 + mse_3 + mse_4
        loss2 = mse_5 + mse_6
        loss3 = mse_7 + mse_8
        loss4 = mse_9 + mse_10
    
    gradients1 = tape.gradient(loss1, net1.trainable_variables)
    gradients2 = tape.gradient(loss2, net2.trainable_variables)
    gradients3 = tape.gradient(loss3, net3.trainable_variables)
    gradients4 = tape.gradient(loss4, net4.trainable_variables)
    optimizer1.apply_gradients(zip(gradients1, net1.trainable_weights))
    optimizer2.apply_gradients(zip(gradients2, net2.trainable_weights))
    optimizer3.apply_gradients(zip(gradients3, net3.trainable_weights))
    optimizer4.apply_gradients(zip(gradients4, net4.trainable_weights))
    
    #train_acc_metric1(eq1_1, Th_hat)    
    #train_acc_metric2(eq6_1, m2_hat)
    #train_acc_metric2(eq8_1, m3_hat)
    #train_acc_metric3(eq9_1, A_hat)
    
    if (epoch + 1) % 100 == 0:
        epochs.append(epoch)
        losses1.append(loss1)
        losses2.append(loss2)
        losses3.append(loss3)
        losses4.append(loss4)
        
        x = np.linspace(Lmin, Lmax, 250)               
        t = np.linspace(0.0, tmax, 100)
        ms_t, ms_x = np. meshgrid(t, x)
        x = np.ravel(ms_x).reshape(-1, 1)
        t = np.ravel(ms_t).reshape(-1, 1)
        pt_Th  = net1(tf. concat([t, x], axis=1))
        pt_SSL = net2(tf. concat([t], axis=1)) * math.pi          # conveert m2 to SSL, [m2/m3]
        pt_Cs  = net3(tf. concat([t], axis=1)) * math.pi/6*rho    # conveert m3 to Solids content [g/L]
        pt_A   = net4(tf. concat([t], axis=1))
        ms_Th  = pt_Th.numpy().reshape(ms_t.shape)
        ms_SSL = pt_SSL.numpy().reshape(ms_t.shape)
        ms_Cs  = pt_Cs.numpy().reshape(ms_t.shape)
        ms_A   = pt_A.numpy().reshape(ms_t.shape)
        
        fig1.clf()
        ax1 = fig1.add_subplot(111, projection='3d')
        ax1.set_zlim([10.0, 30.0])
        ax1.text(0, 0, 1, "epoch:%d" %(epoch + 1), color='black')
        ax1.plot_surface(ms_t, ms_x, ms_Th, cmap=cm.RdYlBu_r, edgecolor='blue', linewidth=0.0003, antialiased=True)
        ax1.set_xlabel('t, h')
        ax1.set_ylabel('L, um')
        ax1.set_zlabel('Th')
        plt.pause(0.1)

        fig2.clf()
        ax2 = fig2.add_subplot(111)
        ax2.set_xlim([0, tmax])
        ax2.plot(t, pt_SSL, '.')
        ax2.set_xlabel('t, h')
        ax2.set_ylabel('SSL, 1/m')
        plt.pause(0.05)        

        fig3.clf()
        ax3 = fig3.add_subplot(111)
        ax3.set_xlim([0, tmax])
        ax3.plot(t, pt_Cs, '.')
        ax3.set_xlabel('t, h')
        ax3.set_ylabel('Cs, g/L')
        plt.pause(0.05) 
        
        fig4.clf()
        ax4 = fig4.add_subplot(111)
        ax4.set_xlim([0, tmax])
        ax4.plot(t, pt_A, '.')
        ax4.set_xlabel('t, h')
        ax4.set_ylabel('Al2O3, g/L')
        plt.pause(0.05)

        fig5.clf()
        ax5 = fig5.add_subplot(111)
        ax5.set_xlim([0, iterations])
        ax5.plot(epochs, losses1)
        ax5.plot(epochs, losses2)
        ax5.plot(epochs, losses3)
        ax5.plot(epochs, losses4)
        ax5.set_yscale('log')
        ax5.set_xlabel('epoch')
        ax5.set_ylabel('loss')
        plt.pause(0.05)
        
        # print training accuracy at the end of each epoch
        #train_acc = train_acc_metric.result()
        #print(f"Training Accuracy   : {train_acc:.3f}")
        elapsed = time.time() - start_time
        #print([epoch+1, elapsed, loss1.numpy(), loss2.numpy(), loss3.numpy(), np.min([Th_hat, Th_min_bc, Th_max_bc]), np.max([Th_hat, Th_min_bc, Th_max_bc]), m2_hat[-1].numpy(), m3_hat[-1].numpy(), A_hat[-1].numpy()])
        print([epoch+1, elapsed, loss2.numpy(), m2_hat[-1].numpy()])
    if ((epoch + 1) % 1000 == 0) and (rewrite_file == True): # save weights every 1000 iterations
        net1.save_weights('net_pbe//net_pbe_20_Th_v10.h5')
        net2.save_weights('net_pbe//net_pbe_20_m2_v10.h5')
        net3.save_weights('net_pbe//net_pbe_20_m3_v10.h5')
        net4.save_weights('net_pbe//net_pbe_20_A_v10.h5')
        
plt.show()
print('Elapsed time = ', elapsed, ' s')

ResourceExhaustedError: {{function_node __wrapped__Mul_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:Mul]

In [ ]:
#n_grid = 30000
#t_pred = tf.constant(np.ones((n_grid, 1))*120, dtype=tf.float32)
#x_pred = tf.constant(tf.random.uniform((n_grid, 1), dtype=tf.float32, minval=0, maxval=Lmax*3))
#x_pred = tf.Variable(Lmin+((1/(1-tf.random.uniform((n_grid, 1), dtype=tf.float32, minval=0.0, maxval=0.99))**0.5-1)/9*(Lmax-Lmin)))
#u_pred = net(tf. concat([t_pred, x_pred], axis=1))
#pyplot.plot(x_pred.numpy(),u_pred.numpy(),'.')
#pyplot.xlabel('L, um')
#pyplot.ylabel('u, #')

In [ ]:
fig1.clf() # Clear the current Figure object
ax1 = fig1.add_subplot(111, projection='3d')
ax1.set_zlim([0, 2e12])
# add text to the image
ax1.text(0, 0, 1, "epoch:%d" %(epoch + 1), color='black')
ax1.plot_surface(ms_t, ms_x, tf.exp(ms_Th), cmap=cm.RdYlBu_r, edgecolor='blue', linewidth=0.0003, antialiased=True)
ax1.set_xlabel('t')
ax1.set_ylabel('L')
ax1.set_zlabel('u')
for angle in range(0, 360):
    ax1.view_init(30, angle)
    plt.draw()
    plt.pause(.001)

In [ ]:
[mse_1, mse_2, mse_3, mse_4, mse_5, mse_6, mse_7, mse_8, mse_9, mse_10]

In [ ]:
ttest=tf.reshape(tf.constant(tf.linspace(0.0, tmax, 100), dtype=tf.float32),[100,1])
net2(tf.concat([ttest], axis=1))# required
#moments = m2m3_func(ttest, x_max2_bc) # it is the slowest line of the whole cycle because of integral for moments
#norm_func(moments[0], m2_min, m2_max)

norm_func(net2(tf.concat([ttest], axis=1)),m2_min, m2_max)
#net2.weights
#m2_field2